# COLAB SETUP

In [1]:
!git clone https://github.com/GeorgeVasile04/llm_lie_detection_black_vs_white_box.git


Cloning into 'llm_lie_detection_black_vs_white_box'...
remote: Enumerating objects: 1555, done.
remote: Counting objects: 100% (511/511), done.
remote: Compressing objects: 100% (401/401), done.
remote: Total 1555 (delta 228), reused 383 (delta 109), pack-reused 1044 (from 1)
Receiving objects: 100% (1555/1555), 91.16 MiB | 39.99 MiB/s, done.
Resolving deltas: 100% (727/727), done.


In [5]:
# Run this cell first if you are executing the notebook in Google Colab.
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab. Installing dependencies...")

    get_ipython().system('pip install -q transformers accelerate datasets scikit-learn pandas seaborn')

    from google.colab import drive
    drive.mount('/content/drive')

    import os, sys

    possible_paths = [
        '/content/drive/MyDrive/llm_lie_detection_black_vs_white_box/Black_vs_White_Lie_Detection/Generalisation_BB_WB',
        '/content/llm_lie_detection_black_vs_white_box/Black_vs_White_Lie_Detection/Generalisation_BB_WB',
        os.path.join(os.getcwd(), 'Black_vs_White_Lie_Detection', 'Generalisation_BB_WB'),
        os.getcwd()
    ]

    base_proj_path = None
    for path in possible_paths:
        if os.path.exists(path) and os.path.exists(os.path.join(path, 'generalization_utils.py')):
            base_proj_path = path
            break

    if base_proj_path:
        os.chdir(base_proj_path)
        print(f"Changed working directory to: {base_proj_path}")
    else:
        print("Could not find project directory. Check your file structure.")

    current_dir   = os.getcwd()
    aligned_dir   = os.path.join(current_dir, '..', 'Aligned_Comparison_BB_WB')
    parent_dir    = os.path.dirname(current_dir)
    project_root  = os.path.dirname(parent_dir)

    for p in [current_dir, aligned_dir, parent_dir, project_root]:
        p = os.path.normpath(p)
        if p not in sys.path:
            sys.path.insert(0, p)

    print(f"sys.path updated.")

    try:
        from huggingface_hub import login
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
        if hf_token:
            login(token=hf_token)
            print("Authenticated with Hugging Face.")
        else:
            login()
    except Exception as e:
        print(f"HF login warning: {e}")

except ImportError:
    IN_COLAB = False
    print("Running locally. Setup skipped.")


Running locally. Setup skipped.


# Imports and Settings

In [6]:
import torch, numpy as np, pandas as pd, pickle, os, time, gc, re
from tqdm.auto import tqdm

from load_data import load_dataset_aligned
from wb_activations import load_model, get_activations_for_dataset
from generalization_utils import (
    evaluate_bb_on_target,
    evaluate_wb_on_target,
    compute_recovered_metrics,
    plot_recovery_heatmap,
)

MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

DATASETS = [
    "commonsense_qa", "race", "arc_easy", "arc_challenge",
    "open_book_qa", "got_cities", "got_sp_en_trans", "got_larger_than", "imdb",
    "amazon_polarity", "ag_news", "dbpedia_14", "rte", "copa", "boolq",
]

# race/copa/boolq use corrected prompts -- classifiers and features have _new suffix
FIXED_MAP         = {"race": "race_new", "copa": "copa_new", "boolq": "boolq_new"}
REVERSE_FIXED_MAP = {v: k for k, v in FIXED_MAP.items()}

E3_DIR      = os.path.join("..", "Aligned_Comparison_BB_WB")
L2_FEAT_DIR = os.path.join(E3_DIR, "bb_elicitation_vectors_lvl_2")
E2_CACHE    = os.path.join("..", "Data_Efficiency_BB_WB", "outputs")
L2_CLF_DIR  = os.path.join(E3_DIR, "classifiers", "level2")
L3_CLF_DIR  = os.path.join(E3_DIR, "classifiers", "level3")
COMPACT_CSV = os.path.join(E3_DIR, "outputs", "compact_comparison_results_final.csv")

METHOD_NAMES = {
    "lr":    "Logistic Regression",
    "dim":   "Difference in Means",
    "lat":   "Linear Artificial Tomography",
    "ccs":   "Contrast-Consistent Search",
    "lda":   "Linear Discriminant Analysis",
    "pca":   "Principal Component Analysis",
    "pca-g": "Grouped PCA",
    "lr-g":  "Grouped Logistic Regression",
}
METHOD_KEYS = {v: k for k, v in METHOD_NAMES.items()}

# 4 best WB methods for the layer-impact study
TARGET_METHODS = ["lr-g", "lda", "lr", "lat"]
FIXED_LAYERS   = [8, 16, 24]

def parse_best_l3(s):
    m = re.match(r"[\d.]+ \((.+), L(\d+)\)", str(s))
    if m:
        return m.group(1), int(m.group(2))
    return None, None

compact_df = pd.read_csv(COMPACT_CSV)
BEST_WB = {}
for _, row in compact_df.iterrows():
    ds = row["Dataset"]
    method_name, layer = parse_best_l3(row["Best L3 WB AUC"])
    if method_name and layer:
        BEST_WB[ds] = {
            "method_key": METHOD_KEYS.get(method_name),
            "layer":      layer,
        }

# Best layer per (dataset, method) from level3_results.csv
_l3_csv = pd.read_csv(os.path.join(E3_DIR, "outputs", "level3_results.csv"))
BEST_LAYER_PER_METHOD = {}
for _, _row in _l3_csv.iterrows():
    _ds  = REVERSE_FIXED_MAP.get(_row["Dataset"], _row["Dataset"])
    _key = METHOD_KEYS.get(_row["Method"])
    if _key in TARGET_METHODS and _ds in DATASETS:
        BEST_LAYER_PER_METHOD[(_ds, _key)] = int(_row["Layer"])

LAYERS_TO_EXTRACT = sorted(
    {info["layer"] for info in BEST_WB.values()} |
    set(BEST_LAYER_PER_METHOD.values()) |
    set(FIXED_LAYERS)
)

print(f"Device: {DEVICE}")
print(f"Best WB info loaded for {len(BEST_WB)} datasets")
print(f"BEST_LAYER_PER_METHOD entries: {len(BEST_LAYER_PER_METHOD)}")
print(f"Unique WB layers needed: {LAYERS_TO_EXTRACT}")


ModuleNotFoundError: No module named 'load_data'

# 1. Load Datasets (all 15)

In [4]:
dataset_dict = {}
print(f"Loading {len(DATASETS)} datasets...")
start = time.time()

for ds_name in tqdm(DATASETS, desc='Step 1 -- Loading datasets'):
    ds_train = load_dataset_aligned(ds_name, split='train',      n_samples=5000)
    ds_val   = load_dataset_aligned(ds_name, split='validation', n_samples=1000)
    df_train = pd.DataFrame(ds_train)
    df_val   = pd.DataFrame(ds_val)
    df = pd.concat([df_train, df_val], ignore_index=True)
    if len(df_train) > 0 and len(df_val) > 0:
        dataset_dict[ds_name] = df
        print(f'  {ds_name}: {len(df):,} samples  (train={len(df_train):,}, val={len(df_val):,})')
    else:
        print(f'  Skipped {ds_name}: missing data.')

print(f'\n Step 1 Complete: {len(dataset_dict)}/{len(DATASETS)} datasets in {time.time()-start:.1f}s')


Loading 15 datasets...


Step 1 -- Loading datasets:   0%|          | 0/15 [00:00<?, ?it/s]

Loading commonsense_qa (split=train) using RepEng logic...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.25M [00:00<?, ?B/s]

...

boolq/test-00000-of-00001.parquet:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9427 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3270 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3245 [00:00<?, ? examples/s]

Loaded 5000 items.
Loading boolq (split=validation) using RepEng logic...
Loaded 1000 items.
  boolq: 6,000 samples  (train=5,000, val=1,000)

 Step 1 Complete: 15/15 datasets in 162.8s


# 2. Load Model

In [5]:
print('Step 2: Loading model...')
start = time.time()
model, tokenizer = load_model(MODEL_NAME, device=DEVICE)
print(f' Step 2 Complete: Model loaded in {time.time()-start:.1f}s')


Step 2: Loading model...
Loading model: meta-llama/Llama-2-7b-chat-hf...


config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

 Step 2 Complete: Model loaded in 39.0s


# 3. Extract WB Activations

Extracts hidden-state activations for all layers required by:
- The best WB probe per dataset (from E3)
- All 4 target methods at their per-dataset best layers
- Fixed layers 8 (Low), 16 (Medium), 24 (High)

Group ids (uid=197609(georg) gid=197609 groups=197609 field) are also stored to support  probe training.


In [6]:
WB_BATCH_SIZE = 64
wb_act_dict = {}

print(f"Step 3: Extracting WB activations (layers={LAYERS_TO_EXTRACT})...")
start = time.time()
outer_pbar = tqdm(DATASETS, desc="Step 3 -- WB Activations")
for ds_name in outer_pbar:
    if ds_name not in dataset_dict:
        continue
    df       = dataset_dict[ds_name]
    df_train = df[df["split"] == "train"].reset_index(drop=True)
    df_val   = df[df["split"] == "validation"].reset_index(drop=True)

    current_batch_size = 28 if ds_name in ("imdb", "race") else WB_BATCH_SIZE
    outer_pbar.set_description(f"WB Activations: {ds_name} (train/val)")

    raw_train = get_activations_for_dataset(df_train, model, tokenizer, device=DEVICE,
                                            batch_size=current_batch_size, show_progress=True)
    raw_val   = get_activations_for_dataset(df_val,   model, tokenizer, device=DEVICE,
                                            batch_size=current_batch_size, show_progress=True)

    # Keep only the layers needed; also store group ids for lr-g probe training
    wb_act_dict[ds_name] = {
        "train": {
            "X": {l: np.array([s["activations"][l] for s in raw_train], dtype=np.float16) for l in LAYERS_TO_EXTRACT},
            "y": np.array([s["label"] for s in raw_train]),
            "g": np.array([s["id"]    for s in raw_train]),
        },
        "validation": {
            "X": {l: np.array([s["activations"][l] for s in raw_val],   dtype=np.float16) for l in LAYERS_TO_EXTRACT},
            "y": np.array([s["label"] for s in raw_val]),
            "g": np.array([s["id"]    for s in raw_val]),
        },
    }
    del raw_train, raw_val
    gc.collect()

print(f"Step 3 Complete: WB activations in RAM for {len(wb_act_dict)} datasets in {time.time()-start:.1f}s")


Step 3: Extracting WB activations (layers=[2, 4, 6, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 22, 23, 24, 25, 26, 27, 29, 30, 31, 32])...


Step 3 -- WB Activations:   0%|          | 0/15 [00:00<?, ?it/s]

Extracting activations (batch_size=64):   0%|          | 0/79 [00:00<?, ?it/s]

Extracting activations (batch_size=64):   0%|          | 0/16 [00:00<?, ?it/s]

Extracting activations (batch_size=28):   0%|          | 0/179 [00:00<?, ?it/s]

...

Extracting activations (batch_size=64):   0%|          | 0/10 [00:00<?, ?it/s]

Extracting activations (batch_size=64):   0%|          | 0/4 [00:00<?, ?it/s]

Extracting activations (batch_size=64):   0%|          | 0/79 [00:00<?, ?it/s]

Extracting activations (batch_size=64):   0%|          | 0/16 [00:00<?, ?it/s]

Step 3 Complete: WB activations in RAM for 15 datasets in 1037.7s


# 4. Load BB Level 2 Features

Priority: `bb_elicitation_vectors_lvl_2/` (newest, includes `_new` for race/copa/boolq) then `Data_Efficiency_BB_WB/outputs/`.

In [7]:
l2_feat_dict = {}

print('Step 4: Loading BB Level 2 features...')
start = time.time()

for ds_name in tqdm(DATASETS, desc='Step 4 -- L2 Features'):
    clf_name = FIXED_MAP.get(ds_name, ds_name)  # race_new / copa_new / boolq_new

    p1 = os.path.join(L2_FEAT_DIR, f'{clf_name}_bb_features_lvl2.pkl')  # _new local
    p2 = os.path.join(L2_FEAT_DIR, f'{ds_name}_bb_features_lvl2.pkl')   # plain local
    p3 = os.path.join(E2_CACHE,    f'{ds_name}_bb_features_efficiency.pkl')  # Data_Efficiency

    if os.path.exists(p1):
        with open(p1, 'rb') as f: data = pickle.load(f)
        X_train, y_train = data['train']
        X_val,   y_val   = data['validation']
        src = 'local (_new)'
    elif os.path.exists(p2):
        with open(p2, 'rb') as f: data = pickle.load(f)
        X_train, y_train = data['train']
        X_val,   y_val   = data['validation']
        src = 'local'
    elif os.path.exists(p3):
        with open(p3, 'rb') as f: data = pickle.load(f)
        X_train, y_train = data['train_full']
        X_val,   y_val   = data['validation']
        src = 'Data_Efficiency'
    else:
        print(f'  No L2 features found for {ds_name} -- skipping.')
        continue

    l2_feat_dict[ds_name] = {
        'X_train': X_train, 'y_train': y_train,
        'X_val':   X_val,   'y_val':   y_val,
    }
    print(f'  {ds_name}: {src}  (train={len(X_train)}, val={len(X_val)})')

print(f'\n Step 4 Complete: L2 features for {len(l2_feat_dict)}/{len(DATASETS)} datasets in {time.time()-start:.1f}s')


Step 4: Loading BB Level 2 features...


Step 4 -- L2 Features:   0%|          | 0/15 [00:00<?, ?it/s]

  commonsense_qa: Data_Efficiency  (train=5000, val=1000)
  race: local (_new)  (train=5000, val=1000)
  arc_easy: Data_Efficiency  (train=4642, val=1000)
  arc_challenge: local (_new)  (train=3260, val=1000)
  open_book_qa: Data_Efficiency  (train=5000, val=1000)
  got_cities: local (_new)  (train=1774, val=606)
  got_sp_en_trans: local (_new)  (train=462, val=116)
  got_larger_than: local (_new)  (train=2364, val=826)
  imdb: Data_Efficiency  (train=5000, val=1000)
  amazon_polarity: Data_Efficiency  (train=5000, val=1000)
  ag_news: Data_Efficiency  (train=5000, val=1000)
  dbpedia_14: Data_Efficiency  (train=5000, val=1000)
  rte: Data_Efficiency  (train=3980, val=1000)
  copa: local (_new)  (train=620, val=200)
  boolq: local (_new)  (train=5000, val=1000)

 Step 4 Complete: L2 features for 15/15 datasets in 0.0s


# 5. Cross-Dataset Evaluation

**L2 (BB):** apply each source classifier on the validation set of all 14 other datasets.

**L3 (WB):** for each source, use the best method + best layer (from `compact_comparison_results_final.csv`) and evaluate on all 14 other datasets.

In [8]:
# Load E3 in-distribution baselines
l2_csv_path = os.path.join(E3_DIR, 'outputs', 'level2_results.csv')
if not os.path.exists(l2_csv_path):
    l2_csv_path = os.path.join(E3_DIR, 'level2_results.csv')  # fallback
df_l2_indist = pd.read_csv(l2_csv_path)

l3_csv_path = os.path.join(E3_DIR, 'outputs', 'level3_results.csv')
df_l3_indist = pd.read_csv(l3_csv_path)

def get_l2_indist(ds_name):
    clf_name = FIXED_MAP.get(ds_name, ds_name)
    row = df_l2_indist[df_l2_indist['Dataset'] == clf_name]
    return {k: row[k].values[0] for k in ['AUC', 'MAP', 'BRP_90']} if not row.empty else None

def get_l3_indist(ds_name, method_key):
    clf_name = FIXED_MAP.get(ds_name, ds_name)
    rows = df_l3_indist[
        (df_l3_indist['Dataset'] == clf_name) &
        (df_l3_indist['Method']  == METHOD_NAMES[method_key])
    ]
    return {k: rows[k].values[0] for k in ['AUC', 'MAP', 'BRP_90']} if not rows.empty else None

print('In-distribution baselines loaded.')
print(f'  L2 CSV: {l2_csv_path}  ({len(df_l2_indist)} rows)')
print(f'  L3 CSV: {l3_csv_path}  ({len(df_l3_indist)} rows)')


In-distribution baselines loaded.
  L2 CSV: ../Aligned_Comparison_BB_WB/outputs/level2_results.csv  (18 rows)
  L3 CSV: ../Aligned_Comparison_BB_WB/outputs/level3_results.csv  (144 rows)


In [9]:
os.makedirs('outputs', exist_ok=True)

print('Step 5a: BB L2 Cross-Dataset Evaluation...')
start = time.time()
l2_results = []
available_l2 = [ds for ds in DATASETS if ds in l2_feat_dict]

for source_ds in tqdm(available_l2, desc='L2 -- source'):
    clf_name = FIXED_MAP.get(source_ds, source_ds)
    clf_path = os.path.join(L2_CLF_DIR, f'{clf_name}_l2_classifier.pkl')
    if not os.path.exists(clf_path):
        print(f'  No L2 classifier for {source_ds} -- skipping.')
        continue
    with open(clf_path, 'rb') as f:
        bundle = pickle.load(f)

    in_dist = get_l2_indist(source_ds)
    if in_dist is None:
        print(f'  No in-dist baseline for {source_ds} -- skipping.')
        continue

    for target_ds in available_l2:
        y_t = l2_feat_dict[target_ds]['y_val']
        X_t = l2_feat_dict[target_ds]['X_val']
        out_dist = in_dist if target_ds == source_ds else evaluate_bb_on_target(bundle, X_t, y_t)
        rec = compute_recovered_metrics(in_dist, out_dist)
        l2_results.append({
            'Source': source_ds, 'Target': target_ds,
            'InDist_AUC':    in_dist['AUC'],   'InDist_MAP':    in_dist['MAP'],   'InDist_BRP_90':    in_dist['BRP_90'],
            'OutDist_AUC':   out_dist['AUC'],  'OutDist_MAP':   out_dist['MAP'],  'OutDist_BRP_90':   out_dist['BRP_90'],
            'Recovered_AUC': rec['Recovered_AUC'],
            'Recovered_MAP': rec['Recovered_MAP'],
            'Recovered_BRP_90': rec['Recovered_BRP_90'],
        })

df_l2_results = pd.DataFrame(l2_results)
df_l2_results.to_csv('outputs/generalisation_l2_results.csv', index=False)
print(f'\n Step 5a Complete: {len(df_l2_results)} rows in {time.time()-start:.1f}s')
print('  -> outputs/generalisation_l2_results.csv')


Step 5a: BB L2 Cross-Dataset Evaluation...


L2 -- source:   0%|          | 0/15 [00:00<?, ?it/s]


 Step 5a Complete: 225 rows in 0.4s
  -> outputs/generalisation_l2_results.csv


In [10]:
print('Step 5b: WB L3 Cross-Dataset Evaluation (best method per source)...')
start = time.time()
l3_results = []
available_l3 = [ds for ds in DATASETS if ds in wb_act_dict and ds in BEST_WB]

for source_ds in tqdm(available_l3, desc='L3 -- source'):
    info       = BEST_WB[source_ds]
    method_key = info['method_key']
    best_layer = info['layer']
    clf_name   = FIXED_MAP.get(source_ds, source_ds)
    clf_path   = os.path.join(L3_CLF_DIR, f'{clf_name}_{method_key}_l3_classifier.pkl')

    if not os.path.exists(clf_path):
        print(f'  No L3 classifier for {source_ds}/{method_key} -- skipping.')
        continue
    with open(clf_path, 'rb') as f:
        probe = pickle.load(f)

    in_dist = get_l3_indist(source_ds, method_key)
    if in_dist is None:
        print(f'  No in-dist L3 baseline for {source_ds}/{method_key} -- skipping.')
        continue

    X_src_train = wb_act_dict[source_ds]['train']['X'][best_layer].astype(np.float32)
    y_src_train = wb_act_dict[source_ds]['train']['y']

    for target_ds in available_l3:
        X_t = wb_act_dict[target_ds]['validation']['X'][best_layer].astype(np.float32)
        y_t = wb_act_dict[target_ds]['validation']['y']
        out_dist = in_dist if target_ds == source_ds else evaluate_wb_on_target(
            probe, method_key, X_src_train, y_src_train, X_t, y_t
        )
        rec = compute_recovered_metrics(in_dist, out_dist)
        l3_results.append({
            'Source': source_ds, 'Target': target_ds,
            'Method': METHOD_NAMES[method_key], 'Layer': best_layer,
            'InDist_AUC':    in_dist['AUC'],   'InDist_MAP':    in_dist['MAP'],   'InDist_BRP_90':    in_dist['BRP_90'],
            'OutDist_AUC':   out_dist['AUC'],  'OutDist_MAP':   out_dist['MAP'],  'OutDist_BRP_90':   out_dist['BRP_90'],
            'Recovered_AUC': rec['Recovered_AUC'],
            'Recovered_MAP': rec['Recovered_MAP'],
            'Recovered_BRP_90': rec['Recovered_BRP_90'],
        })

df_l3_results = pd.DataFrame(l3_results)
df_l3_results.to_csv('outputs/generalisation_l3_results.csv', index=False)
print(f'\n Step 5b Complete: {len(df_l3_results)} rows in {time.time()-start:.1f}s')
print('  -> outputs/generalisation_l3_results.csv')


Step 5b: WB L3 Cross-Dataset Evaluation (best method per source)...


L3 -- source:   0%|          | 0/15 [00:00<?, ?it/s]


 Step 5b Complete: 225 rows in 3.8s
  -> outputs/generalisation_l3_results.csv


# 5c. WB Layer-Impact: 4 Methods × 4 Layer Configs

For each of the 4 best WB methods (Grouped LR, LDA, LR, LAT) and each layer
setting (Best per method, L8, L16, L24), train on each source dataset and
evaluate recovered AUC on all 14 target datasets.


In [11]:
from wb_probes import (
    train_lr_probe, train_lr_g_probe, train_lda_probe, train_lat_probe,
)

LAYER_CONFIGS = ['best'] + FIXED_LAYERS  # ['best', 8, 16, 24]

print('Step 5c: WB Cross-Dataset Evaluation (4 methods x 4 layer configs)...')
start = time.time()
l3_multi_results = []
available_l3_multi = [ds for ds in DATASETS if ds in wb_act_dict]

for method_key in TARGET_METHODS:
    for layer_config in LAYER_CONFIGS:
        config_label = 'Best' if layer_config == 'best' else f'L{layer_config}'

        for source_ds in tqdm(available_l3_multi, desc=f'{method_key}/{config_label}'):
            clf_name = FIXED_MAP.get(source_ds, source_ds)

            if layer_config == 'best':
                layer = BEST_LAYER_PER_METHOD.get((source_ds, method_key))
                if layer is None:
                    continue
                clf_path = os.path.join(L3_CLF_DIR, f'{clf_name}_{method_key}_l3_classifier.pkl')
                if not os.path.exists(clf_path):
                    continue
                with open(clf_path, 'rb') as f:
                    probe = pickle.load(f)
            else:
                layer = layer_config
                if layer not in wb_act_dict[source_ds]['train']['X']:
                    continue
                X_src = wb_act_dict[source_ds]['train']['X'][layer].astype(np.float32)
                y_src = wb_act_dict[source_ds]['train']['y']
                g_src = wb_act_dict[source_ds]['train']['g']
                try:
                    if method_key == 'lr':
                        probe = train_lr_probe(X_src, y_src)
                    elif method_key == 'lr-g':
                        probe = train_lr_g_probe(X_src, y_src, g_src)
                    elif method_key == 'lda':
                        probe = train_lda_probe(X_src, y_src)
                    elif method_key == 'lat':
                        probe = train_lat_probe(X_src)
                    else:
                        continue
                except Exception as e:
                    print(f'  Train failed {method_key}/{source_ds}/L{layer}: {e}')
                    continue

            if layer not in wb_act_dict[source_ds]['train']['X']:
                continue
            X_src_train = wb_act_dict[source_ds]['train']['X'][layer].astype(np.float32)
            y_src_train = wb_act_dict[source_ds]['train']['y']
            X_src_val   = wb_act_dict[source_ds]['validation']['X'][layer].astype(np.float32)
            y_src_val   = wb_act_dict[source_ds]['validation']['y']

            in_dist = evaluate_wb_on_target(probe, method_key, X_src_train, y_src_train, X_src_val, y_src_val)

            for target_ds in available_l3_multi:
                if layer not in wb_act_dict[target_ds]['validation']['X']:
                    continue
                X_t = wb_act_dict[target_ds]['validation']['X'][layer].astype(np.float32)
                y_t = wb_act_dict[target_ds]['validation']['y']
                out_dist = in_dist if target_ds == source_ds else evaluate_wb_on_target(
                    probe, method_key, X_src_train, y_src_train, X_t, y_t
                )
                rec = compute_recovered_metrics(in_dist, out_dist)
                l3_multi_results.append({
                    'Source':           source_ds,
                    'Target':           target_ds,
                    'Method':           METHOD_NAMES[method_key],
                    'Layer_Config':     config_label,
                    'Layer':            layer,
                    'InDist_AUC':       in_dist['AUC'],
                    'OutDist_AUC':      out_dist['AUC'],
                    'Recovered_AUC':    rec['Recovered_AUC'],
                    'Recovered_MAP':    rec['Recovered_MAP'],
                    'Recovered_BRP_90': rec['Recovered_BRP_90'],
                })

df_l3_multi = pd.DataFrame(l3_multi_results)
df_l3_multi.to_csv('outputs/generalisation_l3_multi_results.csv', index=False)
print(f'\nStep 5c Complete: {len(df_l3_multi)} rows in {time.time()-start:.1f}s')
print('  -> outputs/generalisation_l3_multi_results.csv')


Step 5c: WB Cross-Dataset Evaluation (4 methods x 4 layer configs)...


lr-g/Best:   0%|          | 0/15 [00:00<?, ?it/s]

lr-g/L8:   0%|          | 0/15 [00:00<?, ?it/s]

lr-g/L16:   0%|          | 0/15 [00:00<?, ?it/s]

lr-g/L24:   0%|          | 0/15 [00:00<?, ?it/s]

...

lat/Best:   0%|          | 0/15 [00:00<?, ?it/s]

lat/L8:   0%|          | 0/15 [00:00<?, ?it/s]

lat/L16:   0%|          | 0/15 [00:00<?, ?it/s]

lat/L24:   0%|          | 0/15 [00:00<?, ?it/s]


Step 5c Complete: 3600 rows in 421.7s
  -> outputs/generalisation_l3_multi_results.csv


# 6. Recovery Analysis

- **6a** — Heatmaps: BB L2 and WB L3 (G-LR @ L16) normalised recovered AUC
- **6b** — OutDist AUC: BB vs all 4 WB methods across 4 layer settings
- **6c** — Normalised Recovered AUC: 4 WB methods × 4 layer settings + BB
- **6d** — RQ2 direct answer: mean normalised recovered AUC per source and target


In [12]:
import matplotlib.pyplot as plt
import seaborn as sns

print('Step 6a: Heatmaps — BB L2 and WB (G-LR @ L16) normalised recovered AUC...')
os.makedirs('outputs', exist_ok=True)

# ── Load ─────────────────────────────────────────────────────────────────
df_l2_res  = pd.read_csv('outputs/generalisation_l2_results.csv')
df_l3_all  = pd.read_csv('outputs/generalisation_l3_multi_results.csv')

# WB heatmap uses G-LR @ L16: the single WB config with the highest mean
# out-of-distribution AUC across all method+layer combinations.
df_l3_res = df_l3_all[
    (df_l3_all['Method']       == 'Grouped Logistic Regression') &
    (df_l3_all['Layer_Config'] == 'L16')
].copy()

# ── Thresholds: max E3 in-dist AUC per target dataset ────────────────────
_l3_indist = pd.read_csv(os.path.join('..', 'Aligned_Comparison_BB_WB',
                                       'outputs', 'level3_results.csv'))
_FIXED_MAP = {'race': 'race_new', 'copa': 'copa_new', 'boolq': 'boolq_new'}
THRESHOLDS = {}
for _ds in DATASETS:
    _l3_name        = _FIXED_MAP.get(_ds, _ds)
    _sub            = _l3_indist[_l3_indist.Dataset == _l3_name]
    THRESHOLDS[_ds] = _sub.AUC.max()

def add_norm_recovered(df):
    df = df.copy()
    df['Norm_Rec'] = df['OutDist_AUC'] / df['Target'].map(THRESHOLDS)
    return df

df_l2_res = add_norm_recovered(df_l2_res)
df_l3_res = add_norm_recovered(df_l3_res)

ds_order = [ds for ds in DATASETS if ds in df_l2_res["Source"].unique()]

def plot_norm_heatmap(matrix_df, metric_label, title, save_path):
    display_df = matrix_df.clip(upper=1.0)
    fig, ax = plt.subplots(figsize=(15, 12))
    sns.heatmap(
        display_df, annot=True, fmt=".0%",
        cmap="RdYlGn", vmin=0.0, vmax=1.0,
        linewidths=0.4, linecolor="white", ax=ax,
        cbar_kws={"label": metric_label, "shrink": 0.8},
    )
    ax.set_title(title, fontsize=14, pad=15, fontweight="bold")
    ax.set_xlabel("Target Dataset (test)", fontsize=12)
    ax.set_ylabel("Source Dataset (train)", fontsize=12)
    plt.xticks(rotation=45, ha="right", fontsize=9)
    plt.yticks(rotation=0, fontsize=9)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"  Saved -> {save_path}")

# ── Heatmap 1: BB L2 ─────────────────────────────────────────────────────
pivot_l2 = (
    df_l2_res
    .groupby(['Source', 'Target'])['Norm_Rec'].mean()
    .unstack('Target')
    .reindex(index=ds_order, columns=ds_order)
)
plot_norm_heatmap(
    pivot_l2,
    'Normalised Recovered AUC  (OutDist / max E3 in-dist AUC for target)',
    'BB L2 — Cross-Dataset Normalised Recovered AUC',
    'outputs/generalisation_l2_recovered_auc.png',
)

# ── Heatmap 2: WB L3 — G-LR @ L16 ───────────────────────────────────────
ds_order_l3 = [ds for ds in DATASETS if ds in df_l3_res["Source"].unique()]
pivot_l3 = (
    df_l3_res
    .groupby(['Source', 'Target'])['Norm_Rec'].mean()
    .unstack('Target')
    .reindex(index=ds_order_l3, columns=ds_order_l3)
)
plot_norm_heatmap(
    pivot_l3,
    'Normalised Recovered AUC  (OutDist / max E3 in-dist AUC for target)',
    'WB L3 (G-LR @ L16) — Cross-Dataset Normalised Recovered AUC',
    'outputs/generalisation_l3_recovered_auc.png',
)

print('\nStep 6a Complete: 2 heatmaps saved to outputs/')


## 6b. OutDist AUC — BB L2 vs 4 WB Methods × 4 Layer Settings

Absolute out-of-distribution AUC for BB L2 and all 4 WB methods
at 4 layer settings (Best in-distribution layer, L8, L16, L24).
BB appears once per panel as a fixed reference (no layer dependency).
Each point = one off-diagonal (source→target) pair  |  bar = median.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

os.makedirs("outputs", exist_ok=True)

# ── Load data ────────────────────────────────────────────────────────────
df_bb  = pd.read_csv("outputs/generalisation_l2_results.csv")
df_wb  = pd.read_csv("outputs/generalisation_l3_multi_results.csv")

bb_off = df_bb[df_bb.Source != df_bb.Target][["Source","Target","OutDist_AUC"]].copy()
bb_off["Method_Short"] = "BB L2"

wb_off = df_wb[df_wb.Source != df_wb.Target].copy()
METHOD_SHORT = {
    "Grouped Logistic Regression":  "G-LR",
    "Logistic Regression":          "LR",
    "Linear Discriminant Analysis": "LDA",
    "Linear Artificial Tomography": "LAT",
}
wb_off["Method_Short"] = wb_off["Method"].map(METHOD_SHORT)

# ── Print global summary per method × layer config ───────────────────────
bb_mean = bb_off["OutDist_AUC"].mean()
print(f"BB L2 mean OutDist AUC: {bb_mean:.4f}  (210 pairs)")
print()
tbl = (wb_off.groupby(["Layer_Config","Method_Short"])["OutDist_AUC"]
       .mean().unstack("Method_Short").round(4))
tbl["BB L2"] = bb_mean
print("Mean OutDist AUC per method x layer config:")
print(tbl[["BB L2","G-LR","LDA","LR","LAT"]].to_string())

# ── Build long-form frame for seaborn ────────────────────────────────────
LAYER_CONFIGS  = ["Best", "L8", "L16", "L24"]
METHOD_ORDER   = ["BB L2", "G-LR", "LDA", "LR", "LAT"]
PALETTE = {
    "BB L2": "#1f77b4",
    "G-LR":  "#2ca02c",
    "LDA":   "#ff7f0e",
    "LR":    "#9467bd",
    "LAT":   "#d62728",
}

# ── Plot ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(22, 7), sharey=True)
fig.suptitle(
    "6b — OutDist AUC: BB L2 vs 4 WB Methods across 4 Layer Settings\n"
    "(each point = one source\u2192target pair  |  bar = median  |  N=210 per strip)",
    fontsize=13, fontweight="bold",
)

for ax, cfg in zip(axes, LAYER_CONFIGS):
    # WB data for this layer config
    wb_cfg = wb_off[wb_off.Layer_Config == cfg][["Source","Target","OutDist_AUC","Method_Short"]]

    # Stack BB (same for every panel) and WB
    panel_df = pd.concat([
        bb_off.rename(columns={"OutDist_AUC": "OutDist_AUC"}),
        wb_cfg.rename(columns={"OutDist_AUC": "OutDist_AUC"}),
    ], ignore_index=True)

    sns.stripplot(
        data=panel_df,
        x="Method_Short",
        y="OutDist_AUC",
        order=METHOD_ORDER,
        palette=PALETTE,
        alpha=0.35,
        jitter=True,
        size=3,
        ax=ax,
        zorder=1,
    )

    # Median bars + labels
    for i, m in enumerate(METHOD_ORDER):
        vals = panel_df[panel_df.Method_Short == m]["OutDist_AUC"]
        if vals.empty:
            continue
        med = vals.median()
        ax.hlines(med, i - 0.3, i + 0.3, colors="black", linewidths=2.2, zorder=5)
        ax.text(i, med + 0.018, f"{med:.3f}",
                ha="center", fontsize=8.5, fontweight="bold")

    ax.axhline(0.5, color="red",   linestyle=":",  alpha=0.45, linewidth=0.9)
    ax.axhline(1.0, color="green", linestyle="--", alpha=0.20, linewidth=0.9)
    ax.set_title(f"Layer: {cfg}", fontsize=12, fontweight="bold", pad=8)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=20, labelsize=9)
    sns.despine(ax=ax)

axes[0].set_ylabel("OutDist AUC", fontsize=11)
for ax in axes[1:]:
    ax.set_ylabel("")

fig.tight_layout()
fig.savefig("outputs/fair_bb_vs_wb_comparison.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved -> outputs/fair_bb_vs_wb_comparison.png")


In [3]:
import matplotlib.pyplot as plt
import seaborn as sns

print('Step 6c: Normalised Recovered AUC — 4 WB methods × 4 layers + BB...')
os.makedirs('outputs', exist_ok=True)

# ── Load ─────────────────────────────────────────────────────────────────
df_l3_multi = pd.read_csv('outputs/generalisation_l3_multi_results.csv')
df_l2_res   = pd.read_csv('outputs/generalisation_l2_results.csv')

# ── Load level3_results.csv to build per-target thresholds ───────────────
# Threshold for dataset B = max in-distribution AUC ever achieved on B
#   (across all 8 methods x all layers in E3).
# Normalised Recovered AUC = OutDist_AUC(train A, test B) / threshold(B)
# This anchors every evaluation to the best possible performance on that
# target, making results comparable across methods and datasets.
_l3_indist = pd.read_csv(os.path.join('..', 'Aligned_Comparison_BB_WB',
                                       'outputs', 'level3_results.csv'))
_FIXED_MAP = {'race': 'race_new', 'copa': 'copa_new', 'boolq': 'boolq_new'}

THRESHOLDS = {}
_thr_rows  = []
for _ds in DATASETS:
    _l3_name = _FIXED_MAP.get(_ds, _ds)
    _sub     = _l3_indist[_l3_indist.Dataset == _l3_name]
    _best    = _sub.loc[_sub.AUC.idxmax()]
    THRESHOLDS[_ds] = _best.AUC
    _thr_rows.append({'Dataset': _ds, 'L3_name': _l3_name,
                       'Threshold': round(_best.AUC, 4),
                       'Best_method': _best.Method,
                       'Layer': int(_best.Layer)})

_thr_df = pd.DataFrame(_thr_rows)
print('Thresholds (max in-distribution AUC per target dataset from E3):')
print(f"  {'Dataset':20s} {'Threshold':>9}  Best method @ layer")
print('  ' + '-'*60)
for _, r in _thr_df.iterrows():
    print(f"  {r.Dataset:20s} {r.Threshold:>9.4f}  {r.Best_method} @ L{r.Layer}")
print()

# ── Off-diagonal subsets ─────────────────────────────────────────────────
df_wb_off = df_l3_multi[df_l3_multi['Source'] != df_l3_multi['Target']].copy()
df_bb_off = df_l2_res[df_l2_res['Source']   != df_l2_res['Target']].copy()

# ── Compute normalised recovered AUC ─────────────────────────────────────
df_wb_off['Norm_Rec'] = (df_wb_off['OutDist_AUC'] /
    df_wb_off['Target'].map(THRESHOLDS))
df_bb_off['Norm_Rec'] = (df_bb_off['OutDist_AUC'] /
    df_bb_off['Target'].map(THRESHOLDS))

# ── Method labels ────────────────────────────────────────────────────────
METHOD_DISPLAY = {
    'Grouped Logistic Regression': 'Grouped LR',
    'Linear Discriminant Analysis': 'LDA',
    'Logistic Regression': 'LR',
    'Linear Artificial Tomography': 'LAT',
}
METHOD_ORDER    = ['BB L2', 'Grouped LR', 'LDA', 'LR', 'LAT']
PALETTE         = {'BB L2': '#1f77b4', 'Grouped LR': '#2ca02c',
                   'LDA': '#ff7f0e', 'LR': '#9467bd', 'LAT': '#d62728'}
LAYER_CFG_ORDER = ['Best', 'L8', 'L16', 'L24']

df_wb_off['Method_Short'] = df_wb_off['Method'].map(METHOD_DISPLAY)

bb_strip = df_bb_off[['Source', 'Target', 'Norm_Rec']].copy()
bb_strip['Method_Short'] = 'BB L2'

# ── Plot ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(22, 7), sharey=True)
fig.suptitle(
    '6c — Normalised Recovered AUC: 4 WB Methods × 4 Layer Settings + BB\n'
    '(each point = one source\u2192target pair  |  bar = median  |  '
    'Normalised Rec. AUC = OutDist AUC / max in-dist AUC for that target)',
    fontsize=12, fontweight='bold',
)

for ax, config in zip(axes, LAYER_CFG_ORDER):
    wb_cfg = df_wb_off[df_wb_off['Layer_Config'] == config][['Source','Target','Norm_Rec','Method_Short']].copy()

    panel_df = pd.concat([
        bb_strip.rename(columns={'Norm_Rec': 'Norm_Rec'}),
        wb_cfg.rename(columns={'Norm_Rec': 'Norm_Rec'}),
    ], ignore_index=True)

    sns.stripplot(
        data=panel_df,
        x='Method_Short',
        y='Norm_Rec',
        order=METHOD_ORDER,
        palette=PALETTE,
        alpha=0.35,
        jitter=True,
        size=3,
        ax=ax,
        zorder=1,
    )

    for i, m in enumerate(METHOD_ORDER):
        vals = panel_df[panel_df['Method_Short'] == m]['Norm_Rec']
        if vals.empty:
            continue
        med = vals.median()
        ax.hlines(med, i - 0.3, i + 0.3, colors='black', linewidths=2.2, zorder=5)
        ax.text(i, med + 0.02, f'{med:.3f}',
                ha='center', fontsize=8, fontweight='bold')

    ax.axhline(0.5, color='red',   linestyle=':',  alpha=0.45, linewidth=0.9)
    ax.axhline(1.0, color='green', linestyle='--', alpha=0.25, linewidth=0.9)
    ax.set_title(f'Layer: {config}', fontsize=12, fontweight='bold', pad=8)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=20, labelsize=9)
    sns.despine(ax=ax)

axes[0].set_ylabel('Normalised Recovered AUC', fontsize=11)
for ax in axes[1:]:
    ax.set_ylabel('')

fig.tight_layout()
fig.savefig('outputs/generalisation_wb_strip_4configs.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved -> outputs/generalisation_wb_strip_4configs.png')


## 6d. RQ2 Direct Answer — Mean Normalised Recovered AUC per Source and Target

**Normalised Recovered AUC** = OutDist\_AUC(train A → test B) / threshold(B)
where threshold(B) = max in-distribution AUC for dataset B across all E3 methods.

Two breakdowns: per source dataset and per target dataset.


In [15]:
# Off-diagonal only
df_l2_off = df_l2_res[df_l2_res['Source'] != df_l2_res['Target']].copy()
df_l3_off = df_l3_res[df_l3_res['Source'] != df_l3_res['Target']].copy()

# ── Per-source mean normalised recovered AUC ─────────────────────────────
l2_src = df_l2_off.groupby('Source')['Norm_Rec'].mean().rename('BB L2')
l3_src = df_l3_off.groupby('Source')['Norm_Rec'].mean().rename('WB L3 (best)')
src_df = pd.concat([l2_src, l3_src], axis=1)
src_df['WB - BB'] = src_df['WB L3 (best)'] - src_df['BB L2']
src_df = src_df.sort_values('WB - BB', ascending=False)

# ── Per-target mean normalised recovered AUC ─────────────────────────────
l2_tgt = df_l2_off.groupby('Target')['Norm_Rec'].mean().rename('BB L2')
l3_tgt = df_l3_off.groupby('Target')['Norm_Rec'].mean().rename('WB L3 (best)')
tgt_df = pd.concat([l2_tgt, l3_tgt], axis=1)
tgt_df['WB - BB'] = tgt_df['WB L3 (best)'] - tgt_df['BB L2']
tgt_df = tgt_df.sort_values('WB - BB', ascending=False)

# ── Global means ─────────────────────────────────────────────────────────
mean_bb = df_l2_off['Norm_Rec'].mean()
mean_wb = df_l3_off['Norm_Rec'].mean()

print("=" * 62)
print("RQ2: NORMALISED RECOVERED AUC  (OutDist / max E3 in-dist AUC)")
print("=" * 62)
print(f"  Global mean  BB L2       : {mean_bb:.3f}")
print(f"  Global mean  WB L3 (best): {mean_wb:.3f}")
winner = "WB L3" if mean_wb > mean_bb else "BB L2"
print(f"  => {winner} generalises better on average "
      f"(delta = {abs(mean_wb - mean_bb):.3f})")

print("\n--- Per SOURCE dataset ---")
print(src_df.round(3).to_string())

print("\n--- Per TARGET dataset ---")
print(tgt_df.round(3).to_string())

src_df.to_csv("outputs/rq2_bb_vs_wb_generalisation.csv")

# ── Figure: 2 side-by-side bar charts ────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 6))
w = 0.35

# Panel 1 -- per source
x1 = list(range(len(src_df)))
ax1.bar([i - w/2 for i in x1], src_df['BB L2'],
        width=w, label='BB L2', color='#1f77b4')
ax1.bar([i + w/2 for i in x1], src_df['WB L3 (best)'],
        width=w, label='WB L3 (best)', color='#2ca02c')
ax1.axhline(mean_bb, color="#1f77b4", linestyle="--", linewidth=1.2,
            label=f"BB L2 avg ({mean_bb:.2f})")
ax1.axhline(mean_wb, color="#2ca02c", linestyle="--", linewidth=1.2,
            label=f"WB L3 avg ({mean_wb:.2f})")
ax1.set_xticks(x1)
ax1.set_xticklabels(src_df.index, rotation=45, ha="right", fontsize=9)
ax1.set_title("Mean Normalised Recovered AUC per Source Dataset\n"
              "(averaged over 14 target datasets)",
              fontsize=12, pad=10)
ax1.set_xlabel("Source (train) dataset", fontsize=11)
ax1.set_ylabel("Mean Normalised Recovered AUC", fontsize=11)
ax1.set_ylim(0, 1.10)
ax1.axhline(0.5, color="red", linestyle=":", alpha=0.4, linewidth=0.8)
ax1.legend(fontsize=9)
sns.despine(ax=ax1)

# Panel 2 -- per target
x2 = list(range(len(tgt_df)))
ax2.bar([i - w/2 for i in x2], tgt_df['BB L2'],
        width=w, label='BB L2', color='#1f77b4')
ax2.bar([i + w/2 for i in x2], tgt_df['WB L3 (best)'],
        width=w, label='WB L3 (best)', color='#2ca02c')
ax2.axhline(mean_bb, color="#1f77b4", linestyle="--", linewidth=1.2,
            label=f"BB L2 avg ({mean_bb:.2f})")
ax2.axhline(mean_wb, color="#2ca02c", linestyle="--", linewidth=1.2,
            label=f"WB L3 avg ({mean_wb:.2f})")
ax2.set_xticks(x2)
ax2.set_xticklabels(tgt_df.index, rotation=45, ha="right", fontsize=9)
ax2.set_title("Mean Normalised Recovered AUC per Target Dataset\n"
              "(averaged over 14 source datasets)",
              fontsize=12, pad=10)
ax2.set_xlabel("Target (test) dataset", fontsize=11)
ax2.set_ylabel("Mean Normalised Recovered AUC", fontsize=11)
ax2.set_ylim(0, 1.10)
ax2.axhline(0.5, color="red", linestyle=":", alpha=0.4, linewidth=0.8)
ax2.legend(fontsize=9)
sns.despine(ax=ax2)

fig.suptitle(
    "RQ2 — BB L2 vs WB L3 (best method per source)\n"
    "Normalised Recovered AUC = OutDist AUC / max E3 in-dist AUC for target",
    fontsize=13, fontweight="bold",
)
fig.tight_layout()
fig.savefig("outputs/rq2_bb_vs_wb_bar.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved -> outputs/rq2_bb_vs_wb_generalisation.csv  &  rq2_bb_vs_wb_bar.png")
